In [7]:
import numpy as np
import pandas as pd
import faiss
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

In [4]:
embeddings = np.load("../data/processed/customer_support_embeddings.npy")
print(embeddings.shape)

(8469, 384)


In [5]:
embedding_metadata = pd.read_csv("../data/processed/customer_support_embedding_metadata.csv")
print("Metadata shape:", embedding_metadata.shape)

Metadata shape: (8469, 2)


In [6]:
print(embedding_metadata.columns.tolist())
embedding_metadata.head()

['text', 'label']


,text,label
0,product setup issue product please assist bill...,Technical issue
1,peripheral compatibility issue product please ...,Technical issue
2,network problem facing problem product product...,Technical issue
3,account access issue product please assist pro...,Billing inquiry
4,data loss issue product please assist note sel...,Billing inquiry


In [8]:
# Normalize Embeddings

embeddings_normalized = normalize(
    embeddings,
    norm="l2"
).astype("float32")

print("Normalized embedding shape:", embeddings_normalized.shape)

Normalized embedding shape: (8469, 384)


In [9]:
#create fiass index

embedding_dimension = embeddings_normalized.shape[1]

index = faiss.IndexFlatIP(embedding_dimension)
index.add(embeddings_normalized)

In [10]:
print("Number of vectors in index:", index.ntotal)
print("Embedding dimension:", embedding_dimension)

Number of vectors in index: 8469
Embedding dimension: 384


In [11]:
faiss.write_index(index, "../data/processed/customer_support_faiss.index")
print('saved')

saved


In [16]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("model loaded")
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.
Embedding dimension: 384


C:\Users\SURYA\AppData\Local\Temp\ipykernel_25696\3158210772.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


In [19]:
# create knowledge base

knowledge_base = embedding_metadata.copy()
knowledge_base["document_id"] = range(len(knowledge_base))
knowledge_base = knowledge_base[["document_id", "text", "label"]]
print(knowledge_base.shape)

(8469, 3)


In [21]:
knowledge_base.to_csv("../data/processed/customer_support_knowledge_base.csv", index=False)
print('saved')

saved


In [23]:
print(knowledge_base.isnull().sum())
knowledge_base.head()

document_id    0
text           0
label          0
dtype: int64


,document_id,text,label
0,0,product setup issue product please assist bill...,Technical issue
1,1,peripheral compatibility issue product please ...,Technical issue
2,2,network problem facing problem product product...,Technical issue
3,3,account access issue product please assist pro...,Billing inquiry
4,4,data loss issue product please assist note sel...,Billing inquiry


In [25]:
print("Total embeddings :", len(embeddings))
print("Total documents  :", len(knowledge_base))
print("FAISS vectors    :", index.ntotal)

Total embeddings : 8469
Total documents  : 8469
FAISS vectors    : 8469


In [26]:
def semantic_search(query, top_k=5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    query_embedding = normalize(
        query_embedding,
        norm="l2"
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = knowledge_base.iloc[indices[0]].copy()
    results["similarity_score"] = distances[0]

    return results.reset_index(drop=True)

In [27]:
semantic_search("My product is not working")

,document_id,text,label,similarity_score
0,4693,display issue facing problem product product n...,Cancellation request,0.642047
1,163,product recommendation facing problem product ...,Cancellation request,0.638573
2,321,product setup facing problem product product n...,Billing inquiry,0.636931
3,1644,product setup facing problem product product n...,Technical issue,0.634117
4,4808,display issue facing problem product product n...,Refund request,0.628159


In [30]:
results = semantic_search("I have a problem with my payment and billing", top_k=)

for rank, row in results.iterrows():
    print(f"Rank {rank + 1}")
    print(f"Similarity: {row['similarity_score']:.4f}")
    print(f"Label: {row['label']}")
    print(f"Text: {row['text']}")


Rank 1
Similarity: 0.6949
Label: Cancellation request
Text: payment issue issue product please assist please try everything try performed factory reset product hoping would resolve problem help
Rank 2
Similarity: 0.6936
Label: Cancellation request
Text: payment issue issue product please assist thank issue product please assist thank issue performed factory reset product hoping would resolve problem help
Rank 3
Similarity: 0.6910
Label: Billing inquiry
Text: payment issue issue product please assist thank prompt helpful service issue product please assist checked available software updates product none
Rank 4
Similarity: 0.6889
Label: Billing inquiry
Text: payment issue issue product please assist please email message try best resolve thank p sign reviewed troubleshooting steps official support website resolve problem
Rank 5
Similarity: 0.6861
Label: Product inquiry
Text: payment issue issue product please assist thanks product sold not used reviewed troubleshooting steps official supp

In [31]:
results = semantic_search("My new product stopped working after installation, I already tried resetting it, but the problem is still affecting my work. What troubleshooting or support options are available?", top_k=5)

for rank, row in results.iterrows():
    print(f"Rank {rank + 1}")
    print(f"Similarity: {row['similarity_score']:.4f}")
    print(f"Label: {row['label']}")
    print(f"Text: {row['text']}")

Rank 1
Similarity: 0.7400
Label: Technical issue
Text: installation support facing problem product product not turning working fine yesterday respond q something gets performed factory reset product hoping would resolve problem help
Rank 2
Similarity: 0.7302
Label: Refund request
Text: installation support facing problem product product not turning working fine yesterday respond still getting working not sure issue specific device others reported similar problems
Rank 3
Similarity: 0.7110
Label: Billing inquiry
Text: installation support facing problem product product not turning working fine yesterday respond sure works fine already contacted customer support multiple times issue remains unresolved
Rank 4
Similarity: 0.7033
Label: Cancellation request
Text: installation support facing problem product product not turning working fine yesterday respond solution use tried troubleshooting steps mentioned user manual issue persists
Rank 5
Similarity: 0.6996
Label: Technical issue
Text: pro